# SemKey canonical Kaggle smoke
Attach private derived dataset `thestonedape/task-aware-eegtotext` and private checkpoint dataset `thestonedape/glim-zuco-checkpoint`. Kaggle may sanitize `=` from the checkpoint filename, so identity is enforced by requiring exactly one attached `.ckpt` and verifying its SHA-256. Enable the private `GITHUB_TOKEN` secret and Internet. This notebook supports nested Kaggle mounts, validates one identity-ordered real batch through SemKey and GLIM, and does not train.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = 'ac9b66ac01fc28ec1d7c73d372fae6ea18e79e4b'
WORKTREE = '/kaggle/working/SemKey'
GLIM_REPO_URL = 'https://github.com/justin-xzliu/GLIM.git'
GLIM_COMMIT = '1d6cb091559800d0fb2be71902967b70d85d9e6e'
GLIM_WORKTREE = '/kaggle/working/GLIM'
GLIM_CHECKPOINT_SHA256 = '25fcd31d1d6cafc9a0656c50a4916ba6ee106884b269d347284784cc0522c8ba'
assert REPO_URL.startswith('https://github.com/') and 'REPLACE_' not in REPO_URL
assert len(COMMIT) == len(GLIM_COMMIT) == 40
assert all(c in '0123456789abcdef' for c in (COMMIT + GLIM_COMMIT).lower())

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys, torch
from kaggle_secrets import UserSecretsClient
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Add a private Kaggle Secret named GITHUB_TOKEN with read access to the repository'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
for path in (WORKTREE, GLIM_WORKTREE):
    if os.path.exists(path):
        shutil.rmtree(path)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual == COMMIT
subprocess.run(['git', 'clone', GLIM_REPO_URL, GLIM_WORKTREE], check=True)
subprocess.run(['git', '-C', GLIM_WORKTREE, 'checkout', '--detach', GLIM_COMMIT], check=True)
actual_glim = subprocess.check_output(['git', '-C', GLIM_WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_glim == GLIM_COMMIT
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'lightning==2.4.0', 'torchmetrics==1.3.1', 'einops==0.8.0', 'timm==0.9.16', 'transformers==4.52.0'], check=True)

In [ ]:
manifest_paths = glob.glob('/kaggle/input/**/metadata/shard_manifest.json', recursive=True)
assert len(manifest_paths) == 1, manifest_paths
dataset_root = os.path.dirname(os.path.dirname(manifest_paths[0]))
subprocess.run([sys.executable, os.path.join(WORKTREE, 'kaggle', 'smoke_input.py'), '--dataset-root', dataset_root, '--batch-size', '1'], check=True)
semkey_report_path = '/kaggle/working/semkey_batch1_report.json'
subprocess.run([sys.executable, os.path.join(WORKTREE, 'kaggle', 'smoke_semkey_sharded_loader.py'), '--dataset-root', dataset_root, '--phase', 'val', '--output', semkey_report_path], check=True)
checkpoint_paths = glob.glob('/kaggle/input/**/*.ckpt', recursive=True)
assert len(checkpoint_paths) == 1, 'Attach the official GLIM checkpoint as a private Kaggle input'
checkpoint = checkpoint_paths[0]
digest_state = hashlib.sha256()
with open(checkpoint, 'rb') as handle:
    for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
        digest_state.update(block)
digest = digest_state.hexdigest()
assert digest == GLIM_CHECKPOINT_SHA256, (digest, GLIM_CHECKPOINT_SHA256)
glim_report_path = '/kaggle/working/glim_batch1_report.json'
subprocess.run([sys.executable, os.path.join(WORKTREE, 'kaggle', 'smoke_glim_sharded_representation.py'), '--dataset-root', dataset_root, '--glim-root', GLIM_WORKTREE, '--checkpoint', checkpoint, '--phase', 'val', '--prompt-mode', 'canonical', '--output', glim_report_path], check=True)
semkey_report = json.load(open(semkey_report_path, encoding='utf-8'))
glim_report = json.load(open(glim_report_path, encoding='utf-8'))
assert semkey_report['status'] == 'pass' and glim_report['status'] == 'glim_batch1_pass'
assert semkey_report['sample_id'] == glim_report['sample_id']
assert semkey_report['source_dataframe_row_index'] == glim_report['source_dataframe_row_index']
assert semkey_report['prompt'][0] == glim_report['canonical_prompt']
assert glim_report['checkpoint_sha256'] == GLIM_CHECKPOINT_SHA256
subprocess.run([sys.executable, '-m', 'py_compile', os.path.join(WORKTREE, 'data', 'datamodule.py'), os.path.join(WORKTREE, 'model', 'semkey_parallel.py')], check=True)
print({'dataset_root': dataset_root, 'sample_id': semkey_report['sample_id'], 'source_dataframe_row_index': semkey_report['source_dataframe_row_index'], 'canonical_prompt': glim_report['canonical_prompt'], 'checkpoint_sha256': digest})
for path in (WORKTREE, GLIM_WORKTREE):
    if os.path.exists(path):
        shutil.rmtree(path)
print('Kaggle canonical SemKey/GLIM same-batch smoke: PASS')